# Setup

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

from pathlib import Path
import requests

In [ ]:
from matplotlib.patches import Rectangle
from scipy.ndimage import convolve

In [ ]:
import matplotlib as mpl

plt.style.use('dark_background')
mpl.rcParams['legend.frameon'] = False

In [ ]:
plt.rcParams.update({
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "lines.markersize": 7,
})

In [ ]:
def remove_ticks(ax):
    ax.set_xticks([])
    ax.set_yticks([])

In [ ]:
# folder to output generated figures
path_figures = Path('./figures')
path_figures.mkdir(exist_ok=True)

# figure parameters
DPI = 150 # this does not apply to svg files

def save_figure(filename, extension='svg'):
    plt.savefig(f"{path_figures}/{filename}.{extension}", transparent=True, dpi=DPI)

# Load data

In [ ]:
# define names and download links for genomic data
path_downloads = Path("./data")
path_downloads.mkdir(exist_ok=True)

dict_org_id = {
    "bacteriophage lambda": "NC_001416.1",
    "Staphylococcus aureus": "NC_007795.1",
    "Homo sapiens Chr. 1": "NC_000001.11",
    "Escherichia coli": "NC_000913.3",
    "Homo sapiens mit.": "NC_001807.4"
}

# here are the rest of the human chromosomes
#https://www.ncbi.nlm.nih.gov/datasets/genome/GCF_000001405.40/

In [ ]:
# download data
for name, org_id in dict_org_id.items():
    filename = path_downloads / f"{"_".join(name.split())}.fasta"

    # skip re-downloads
    if filename.exists():
        continue

    # get download url from NCBI and save to disk
    url = f"https://www.ncbi.nlm.nih.gov/search/api/download-sequence/?db=nuccore&id={org_id}"
    response = requests.get(url, timeout=30)
    response.raise_for_status()

    filename.write_bytes(response.content)

In [ ]:
def parse_fasta(path_fasta):
    """
    Parse a FASTA file and return a tuple (header, sequence)
    """
    with open(path_fasta) as inpt:
        data = inpt.read().split('\n')
        header = data[0]
        sequence = ''.join(data[1:])
    return (header, sequence)

In [ ]:
# load genomic data as dict[org_name] = (header, sequence)
dict_seqs = {}
for path_fasta in path_downloads.glob("*.fasta"):
    name = path_fasta.stem.replace('_', ' ')
    header, sequence = parse_fasta(path_fasta)
    dict_seqs[name] = (header, sequence)

In [ ]:
for name in dict_seqs.keys():
    print(f">{name}")

# Example 1: ATCG content

In [ ]:
# print global ATCG ratios for each genome
for name, (header, sequence) in dict_seqs.items():
    print(name)
    for i in ['A', 'T', 'C', 'G']:
        print("{}: {:2.1%}".format(i, sequence.count(i)/len(sequence)))
    print()

In [ ]:
def compute_symbol_ratio(subsequence, list_symbols):
    """
    Compute the proportion of a sequence matching any element in list_symbols.
    """
    length_subseq = len(subsequence)
    total_count = 0
    for i in list_symbols:
        symbol_count = subsequence.count(i)
        total_count += symbol_count
    return total_count/length_subseq

In [ ]:
def plot_ratio(sequence, ax=None):
    """
    Generate figure of AC/GC enrichment along a sequence
    """
    if ax is None:
        fig, ax = plt.subplots()

    # set up 50 windows per genome
    N = len(sequence)
    window_length = len(sequence) // 50

    # store values per window
    list_GC_ratio = []
    list_AT_ratio = []
    list_xticks = []

    # remove one window to ensure all have equal size
    list_windows = list(range(0, N, window_length))[:-1]
    
    for i in list_windows:
        subsequence = sequence[i:i+window_length]

        # these are complementary so this is a bit unnecessary (sue me)
        ratio_GC = compute_symbol_ratio(subsequence, ['G', 'C'])
        ratio_AT = compute_symbol_ratio(subsequence, ['A', 'T'])
        
        list_GC_ratio.append(ratio_GC)
        list_AT_ratio.append(ratio_AT)
        list_xticks.append(i)

    # plot data
    ax.plot(list_xticks, list_GC_ratio, label='G+C')
    ax.plot(list_xticks, list_AT_ratio, label='A+T')
    ax.set_ylim([0.2, 0.8])
    ax.locator_params(axis='x', nbins=6)
    ax.ticklabel_format(axis="x", style="scientific", scilimits=(0, 0))
    ax.legend()

In [ ]:
fig, axarr = plt.subplots(nrows=2, ncols=2, figsize=(8,6))

list_orgs = ['bacteriophage lambda',
             'Homo sapiens Chr. 1',
             'Staphylococcus aureus',
             'Homo sapiens mit.']

for i, name in enumerate(list_orgs):
    header, sequence = dict_seqs[name]
    ax = axarr.flatten()[i]
    plot_ratio(sequence, ax)
    ax.set_title(name)

plt.tight_layout()
save_figure(f"GC_ratio_ma_figure")

# Example 2: Triplet enrichment

In [ ]:
def compute_kmer_prob(row):
    """
    Calculate p(kmer), given the background random model
    """
    kmer = row.kmer
    prob=1
    for i in kmer:
        prob = prob * dict_ratios[i]
    return prob

In [ ]:
def compute_kmer_odds_table(sequence, kmer):
    """
    Generate table of kmer odds table
    """
    # list kmers (of size k=kmer) in sequence
    list_kmers = [sequence[i:i+kmer] for i in range(len(sequence)-kmer+1)]

    # dataframe of kmer counts
    # WARN: this assumes that each genome contains all possible triplets!
    df_kmers = (pd.DataFrame(
            data=pd.Series(list_kmers)
                .value_counts()
                .reset_index()
                .values, columns=['kmer', 'counts']))

    # add probability under random model
    df_kmers['prob'] = df_kmers.apply(compute_kmer_prob, axis=1)

    # add expected number of kmers
    df_kmers['expected'] = len(list_kmers)*df_kmers['prob']

    # compute odds ratio between found and expected
    df_kmers['odds_ratio'] = df_kmers['counts'] / df_kmers['expected']

    # sort by enrichment (odds ratio)
    df_kmers.sort_values(by=['odds_ratio'], inplace=True, ascending=False)
    
    return df_kmers

In [ ]:
dict_logodds = {}

for org in list(dict_seqs.keys()):
    print(org)
    dict_logodds[org] = {}

    # fetch genomic sequence
    sequence = dict_seqs[org][1].replace('N','')

    # get nucleotide ratios for random background model
    dict_ratios = {char: round(sequence.count(char)/len(sequence),3)
                   for char in ['A', 'T', 'C', 'G']}
    
    # compute dataframes of log-odds ratios
    for kmer in range(2,4):
        df_kmers = compute_kmer_odds_table(sequence, kmer)
        df_kmers.name = org
        dict_logodds[org][kmer] = df_kmers

In [ ]:
df_ecoli = dict_logodds['Escherichia coli'][3]
df_phage = dict_logodds['bacteriophage lambda'][3]
df_staph = dict_logodds['Staphylococcus aureus'][3]
df_homo = dict_logodds['Homo sapiens Chr. 1'][3]
df_mito = dict_logodds['Homo sapiens mit.'][3]

In [ ]:
df_mito.head(5)

In [ ]:
print(df_mito.head(5).to_markdown(index=False))

In [ ]:
df_mito.tail(5)

In [ ]:
print(df_mito.tail(5).to_markdown(index=False))

In [ ]:
def plot_compare_triplets(df_1, df_2, ax=None):
    """
    Make plot comparing triplet usage between two kmer tables
    """
    # join tables
    merged = pd.merge(df_1, df_2, on="kmer")

    if ax is None:
        fig, ax = plt.subplots()
    
    # calculate limits for- and plot the 1:1 diagonal
    lims = [min(merged["odds_ratio_x"].min(), merged["odds_ratio_y"].min()) - 0.1,
            max(merged["odds_ratio_x"].max(), merged["odds_ratio_y"].max()) + 0.1]
    ax.plot(lims, lims, "--", color="gray", label="1:1")

    # graph data for each triplet vs triplet comparison
    for _, row in merged.iterrows():
        # coordinates and color
        x = row["odds_ratio_x"]
        y = row["odds_ratio_y"]
        color = "green" if y > x else "red"
    
        # vertical line to 1:1 diagonal
        ax.plot([x, x], [x, y], color=color,
                alpha=0.4, linewidth=1)
    
        # point of data
        ax.scatter(x, y, color=color,
                   s=40, zorder=3)

    # limits and aspect
    ax.set_xlim(lims)
    ax.set_ylim(lims)
    ax.set_aspect("equal", adjustable="box")

    # labels and legend
    ax.set_xlabel(f"{df_1.name} odds ratio")
    ax.set_ylabel(f"{df_2.name} odds ratio")
    ax.legend()
    
    plt.tight_layout()

In [ ]:
fig, axarr = plt.subplots(ncols=2, figsize=(8,4))

ax = axarr[0]
plot_compare_triplets(df_ecoli, df_phage, ax)

ax = axarr[1]
plot_compare_triplets(df_ecoli, df_homo, ax)

save_figure('triplet_enrichment_comparison')

In [ ]:
fig, axarr = plt.subplots(ncols=2, figsize=(8,4))

ax = axarr[0]
plot_compare_triplets(df_mito, df_staph, ax)

ax = axarr[1]
plot_compare_triplets(df_mito, df_homo, ax)

save_figure('triplet_enrichment_comparison_2')

# Example 3: Repetitive patterns

In [ ]:
def compute_naive_dotplot(seq_1, seq_2):
    """
    Compute binary dotplot, reflecting inter-sequence identities
    """
    arr_seq1 = np.array(list(seq_1))
    arr_seq2 = np.array(list(seq_2))
    mat_matches = arr_seq1[:, None] == arr_seq2[None, :]
    return mat_matches

In [ ]:
# this is an arbitrary segment of the human chr. 1 which features interesting repeats
sample_sequence = dict_seqs['Homo sapiens Chr. 1'][1][130000:140000]

# some more interesting repeats
# sample_sequence = dict_seqs['Homo sapiens Chr. 1'][1][40000:50000]
# sample_sequence = dict_seqs['Homo sapiens Chr. 1'][1][50000:60000]
# sample_sequence = dict_seqs['Homo sapiens Chr. 1'][1][60000:70000]

In [ ]:
# compute NxN dotplot
mat_matches = compute_naive_dotplot(sample_sequence, sample_sequence)

# number of identical matches in NxN self-comparison
mat_matches.sum()

In [ ]:
fig, ax = plt.subplots(figsize=(7,7))

# plot NxN comparison dotplot
ax.imshow(mat_matches, cmap='gray', origin='lower')

# highlight square on area of next analyses
rect = Rectangle(
    (6000, 6000), 1000, 1000,
    linewidth=0.5,
    edgecolor='red',
    facecolor='none')
ax.add_patch(rect)

plt.tight_layout()
save_figure(f"dotplot_full", extension='webp')

In [ ]:
# let us focus on the subsegment highlighted above
target_sequence = sample_sequence[6000:7000]
target_mat = mat_matches[6000:7000,6000:7000]

print(target_sequence)

In [ ]:
# define a kernel to detect diagonal lines
kernel = np.eye(11)
kernel[5, 5] = 11
kernel /= kernel.sum()

# generate matrix of "diagonal scores"
S_diag = convolve(target_mat, kernel)

In [ ]:
# dotplot repeat detection figure
fig, axarr = plt.subplots(ncols=2, figsize=(6,3))

ax = axarr[0]
ax.imshow(target_mat, origin='lower')
ax.set_xlabel('raw dotplot')

ax = axarr[1]
ax.imshow(S_diag, origin='lower')
ax.set_xlabel('convolved matrix')

# aesthetic polish
for ax in axarr:
    remove_ticks(ax)
    ax.xaxis.set_label_position('top')

plt.tight_layout()
save_figure(f"dotplot_detection", extension='webp')

In [ ]:
# focus on the regularly repetitive area
target_S_diag = S_diag[200:,200:]

# verticalize diagonals by offsetting each row
B = np.array([
    np.roll(row, -i)
    for i, row in enumerate(target_S_diag)
])

# separate full diagonal score matrix into halves
upper_triangle = np.triu(np.fliplr(B))
lower_triangle = np.tril(np.fliplr(B))

In [ ]:
fig, axarr = plt.subplots(nrows=3, ncols=3, figsize=(8,6.5),
                          sharex=True, sharey='row',
                          gridspec_kw={'height_ratios': [2,1,1]}, dpi=DPI)

for i, mat in enumerate([B, upper_triangle, lower_triangle]):
    # show verticalized matrices
    ax = axarr[0, i]
    ax.imshow(mat, origin='lower')

    # vertical mean
    ax = axarr[1, i]
    ax.plot(mat.mean(axis=0))
    ax.set_ylim([0,  0.45])

    # vertical mean + binary filter
    ax = axarr[2, i]
    ax.plot(mat.sum(axis=0) > 10)
    ax.set_ylim(bottom=0.0)


# remove ticks in all axes
for ax in axarr.flatten():
    remove_ticks(ax)

# add annotation labels
for i, ylabel in enumerate(['Matrix', 'Average', 'Boolean filter']):
    axarr[i, 0].set_ylabel(ylabel)

for i, xlabel in enumerate(['Full matrix', 'Lower diagonal', 'Upper diagonal']):
    axarr[0, i].set_xlabel(xlabel)
    axarr[0, i].xaxis.set_label_position('top')

plt.tight_layout()
save_figure('repeat_matrix_analyses', extension='webp')

In [ ]:
# we can combine the two halves by inverting the order to align the offsets
joint_signal = np.sum(upper_triangle, axis=0) + np.sum(lower_triangle, axis=0)[::-1]
plt.plot(joint_signal)

In [ ]:
# center signal
x0 = joint_signal - joint_signal.mean()

# consider periodicities between 1 and 100
periods = np.arange(1, 100)
# compute autocorrelation scores
scores = [np.dot(x0[:-p], x0[p:])
          for p in periods]

# identify the dominating period by its maximum score
period = np.argmax(scores) + 1

In [ ]:
fig, ax = plt.subplots(figsize=(6,4))

# show periodicity vs autocorrelation score in bars
ax.bar(periods, scores, width=0.8, alpha=0.7)

# annotate dominating periodicity
ax.axvline(
    period,
    color="red",
    linestyle="--",
    label=f"Estimated period = {period}"
)

# labels and polish
ax.set_xlabel("Period (lag)")
ax.set_ylabel("Autocorrelation score")
ax.set_yticks([])
ax.legend()

plt.tight_layout()
save_figure('lagged_dotproduct')

In [ ]:
fig, axarr = plt.subplots(nrows=2, ncols=2, figsize=(6,4))

# compute FFT for raw (centered) and binary signal
for i, x0 in enumerate([joint_signal - joint_signal.mean(),
                        joint_signal > 10]):
    
    ax = axarr[0,i]
    ax.plot(x0)
    ax.set_ylim(bottom=0)
    ax.set_yticks([])
    ax.set_xlabel('Absolute index')

    ax = axarr[1,i]
    
    # compute the real-valued Discrete Fourier Transform 
    X = np.fft.rfft(x0)
    # compute the circular autocorrelation (inverse)
    corr = np.fft.irfft(np.abs(X)**2)

    # identify the dominant period by its maximum score
    # excluding lag 0 (which is trivially maximal)
    period = np.argmax(corr[1:]) + 1

    # plot data
    ax.plot(corr)
    ax.axvline(x=period, color='red', linestyle='--', alpha=0.5,
               label=f"Est. period = {period}")
    ax.set_yticks([])
    ax.set_ylim(bottom=0)
    ax.set_xlabel('Lag (residues)')
    ax.legend()

# annotations labels
axarr[0,0].set_title('Raw data')
axarr[0,1].set_title('Binary filter')
axarr[0,0].set_ylabel('Signal')
axarr[1,0].set_ylabel('Autocorrelation')

plt.tight_layout()
save_figure('fourier_transforms')

In [ ]:
P = period
print(f"Estimated period: {period}", end='\n\n')
for i in range(1, len(target_sequence), P):
    print(target_sequence[i:i+P])